# Repository note
This is a code-only copy of an exploratory notebook. Saved outputs were removed for public sharing. The required CSV datasets are not included; update the local/Colab paths in the cells before running. The original execution environment and full dataset provenance were not supplied, so a fresh run is not guaranteed to reproduce prior results.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import pandas as pd
import numpy as np

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
train_df = pd.read_csv(r"C:\Users\Gpu\Desktop\Network Steganography\train_dataset.csv")
test_df  = pd.read_csv(r"C:\Users\Gpu\Desktop\Network Steganography\test_dataset.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

In [ ]:
leakage_columns = ["src_port", "dst_port", "flow_id", "timestamp"]

train_df = train_df.drop(columns=[c for c in leakage_columns if c in train_df.columns])
test_df = test_df.drop(columns=[c for c in leakage_columns if c in test_df.columns])

In [ ]:
for col in train_df.columns:
    print(col)

In [ ]:
# Features
X_train = train_df.drop(columns=["label_binary","label_technique","label_protocol_enc"])

# Labels
y_train_binary = train_df["label_binary"]
y_train_tech = train_df["label_technique"]
y_train_proto = train_df["label_protocol_enc"]


# Features
X_test = test_df.drop(columns=["label_binary","label_technique","label_protocol_enc"])

# Labels
y_test_binary = test_df["label_binary"]
y_test_tech = test_df["label_technique"]
y_test_proto = test_df["label_protocol_enc"]

In [ ]:
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("Binary distribution:")
print(y_train_binary.value_counts())

print("Technique distribution:")
print(y_train_tech.value_counts())

print("Protocol distribution:")
print(y_train_proto.value_counts())

In [ ]:
import torch

# Convert features
X_train = torch.tensor(X_train.values, dtype=torch.float32)
X_test  = torch.tensor(X_test.values, dtype=torch.float32)

# Convert labels
y_train_binary = torch.tensor(y_train_binary.values, dtype=torch.float32)
y_test_binary  = torch.tensor(y_test_binary.values, dtype=torch.float32)

y_train_tech = torch.tensor(y_train_tech.values, dtype=torch.long)
y_test_tech  = torch.tensor(y_test_tech.values, dtype=torch.long)

y_train_proto = torch.tensor(y_train_proto.values, dtype=torch.long)
y_test_proto  = torch.tensor(y_test_proto.values, dtype=torch.long)

# Fix binary label shape
y_train_binary = y_train_binary.unsqueeze(1)
y_test_binary  = y_test_binary.unsqueeze(1)

In [ ]:
print(X_train.shape)
print(y_train_binary.shape)
print(y_train_tech.shape)
print(y_train_proto.shape)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(
    X_train,
    y_train_binary,
    y_train_tech,
    y_train_proto
)

test_dataset = TensorDataset(
    X_test,
    y_test_binary,
    y_test_tech,
    y_test_proto
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64)

In [ ]:
for batch in train_loader:
    x,b,t,p = batch
    print(x.shape,b.shape,t.shape,p.shape)
    break

In [ ]:
class AttentionBlock(nn.Module):

    def __init__(self,embed_dim=128,heads=8):
        super().__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim,
            heads,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(embed_dim)

        self.ff = nn.Sequential(
            nn.Linear(embed_dim,256),
            nn.ReLU(),
            nn.Linear(256,embed_dim)
        )

        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self,x):

        attn_output,attn_weights = self.attention(x,x,x)

        x = self.norm1(x+attn_output)

        ff_out = self.ff(x)

        x = self.norm2(x+ff_out)

        return x,attn_weights

In [ ]:
class FeatureTokenTransformer(nn.Module):

    def __init__(self,num_features=225,embed_dim=128):

        super().__init__()

        self.embedding = nn.Linear(1,embed_dim)

        self.blocks = nn.ModuleList([
            AttentionBlock(embed_dim) for _ in range(4)
        ])

        self.shared = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim,128),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.binary_head = nn.Linear(128,1)
        self.tech_head = nn.Linear(128,4)
        self.proto_head = nn.Linear(128,4)

    def forward(self,x):

        x = x.unsqueeze(-1)

        x = self.embedding(x)

        attention_maps = []

        for block in self.blocks:
            x,attn = block(x)
            attention_maps.append(attn)

        x = torch.mean(x,dim=1)

        shared = self.shared(x)

        binary = self.binary_head(shared)
        tech = self.tech_head(shared)
        proto = self.proto_head(shared)

        return binary,tech,proto,attention_maps

In [ ]:
# Binary class weight based on training distribution
pos_weight = torch.tensor([2769/8309]).to(device)

# Loss functions
binary_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

tech_loss = nn.CrossEntropyLoss()

proto_loss = nn.CrossEntropyLoss()

In [ ]:
def multitask_loss(pb, pt, pp, yb, yt, yp):

    l1 = binary_loss(pb, yb)

    l2 = tech_loss(pt, yt)

    l3 = proto_loss(pp, yp)

    return l1 + l2 + l3

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = FeatureTokenTransformer().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)

In [ ]:
for epoch in range(50):

    model.train()

    total_loss = 0

    for x,yb,yt,yp in train_loader:

        x = x.to(device)
        yb = yb.to(device)
        yt = yt.to(device)
        yp = yp.to(device)

        optimizer.zero_grad()

        pb, pt, pp, _ = model(x)

        loss = multitask_loss(pb, pt, pp, yb, yt, yp)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print("Epoch:", epoch, "Loss:", total_loss/len(train_loader))

In [ ]:
model.eval()

pred_binary = []
true_binary = []

pred_tech = []
true_tech = []

pred_proto = []
true_proto = []

with torch.no_grad():

    for x, yb, yt, yp in test_loader:

        x = x.to(device)

        pb, pt, pp, _ = model(x)

        pb = torch.sigmoid(pb)

        pb = (pb > 0.5).cpu()

        pt = torch.argmax(pt, 1).cpu()
        pp = torch.argmax(pp, 1).cpu()

        pred_binary.extend(pb.numpy())
        true_binary.extend(yb.numpy())

        pred_tech.extend(pt.numpy())
        true_tech.extend(yt.numpy())

        pred_proto.extend(pp.numpy())
        true_proto.extend(yp.numpy())

In [ ]:
print("Binary Accuracy:",accuracy_score(true_binary,pred_binary))
print("Technique Accuracy:",accuracy_score(true_tech,pred_tech))
print("Protocol Accuracy:",accuracy_score(true_proto,pred_proto))

In [ ]:
cm = confusion_matrix(true_binary,pred_binary)

plt.figure(figsize=(6,5))

sns.heatmap(cm,annot=True,fmt="d",cmap="Blues")

plt.title("Binary Confusion Matrix")

plt.show()

In [ ]:
labels = ["Binary","Technique","Protocol"]

acc = [
accuracy_score(true_binary,pred_binary),
accuracy_score(true_tech,pred_tech),
accuracy_score(true_proto,pred_proto)
]

plt.bar(labels,acc)

plt.title("Task Accuracy")

plt.show()

In [ ]:
# Save trained model
torch.save(model.state_dict(), "steganography_transformer_model.pth")

In [ ]:
config = {
    "num_features":225,
    "embed_dim":128,
    "heads":8,
    "layers":4
}

with open("model_config.json","w") as f:

    json.dump(config,f)